# ARGUS — Phishing Detection Training (PhishTank + Tranco)

End-to-end: raw URLs → **your** `src/features` pipeline → DT / RF / LR / SVM → evaluate → SHAP → save.

Aligned to: **DEC-002** (DT/RF/LR/SVM), **DEC-003** (PhishTank + Tranco), **DEC-004** (F1 primary, ROC-AUC tie-break), **DEC-006** (stratified 5-fold CV on train; test touched once).

> **Note:** notebooks may import *from* `src/`, but `src/` must never import from notebooks. This notebook is exploratory; productionising it into `scripts/train.py` is Phase 5.

> **⚠️ Validity warning (read before trusting results):** PhishTank gives **full URLs with paths**; Tranco gives **bare domains**. If you feed both raw, the model can learn *"has a path ⇒ phishing"* — a collection artifact, not phishing. The `STRIP_TO_HOSTNAME` toggle below is the conservative mitigation; the alternative (better, more work) is to collect real legitimate URLs with paths. Document whichever you choose in `DATASET_NOTES.md`.

## 0. Setup — make `src/` importable and pin the seed

In [ ]:
import sys, json, time, hashlib
from pathlib import Path

# Find repo root (the dir that contains src/) whether we run from repo root or notebooks/
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src" / "features" / "pipeline.py").exists(), f"src not found from {REPO_ROOT}"
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

import numpy as np
import pandas as pd

RANDOM_STATE = 42  # Iron Law #6 — reproducibility everywhere
np.random.seed(RANDOM_STATE)

from src.features.pipeline import extract_features  # your Phase-3 pipeline

## 1. Config — every knob in one place (no magic numbers buried in code)

In [ ]:
RAW_DIR       = REPO_ROOT / "data" / "raw"
CACHE_DIR     = REPO_ROOT / "data" / "cache";     CACHE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = REPO_ROOT / "data" / "processed"; PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR    = REPO_ROOT / "models" / "production"; MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Corpus size & balance ---
N_PER_CLASS = 5000        # sample this many of EACH class (raise once it works)

# --- Feature-extraction tiers (speed vs richness) ---
FETCH_HTML     = False    # True = live-fetch each page (slow; many PhishTank URLs are dead)
INCLUDE_DOMAIN = False    # True = WHOIS/DNS per URL (slow). Start lexical-only.
MAX_WORKERS    = 16       # thread pool for extraction

# --- Source-artifact bias mitigation (see warning at top) ---
STRIP_TO_HOSTNAME = False  # True = reduce BOTH classes to scheme://host (kills path bias, loses path signal)

# --- Tranco list (legit). The pip package pins a list ID for reproducibility ---
TRANCO_LIST_ID = None     # None = latest; or set a specific ID string you cite in the thesis

FORCE_REEXTRACT = False   # True = ignore the feature cache and recompute
print(f"tiers: lexical=always html={FETCH_HTML} domain={INCLUDE_DOMAIN} | {N_PER_CLASS}/class | strip_host={STRIP_TO_HOSTNAME}")

## 2. Load the two sources → one `[url, label]` corpus
PhishTank → label **1 (phishing)**, Tranco → label **0 (legit)**. Labels are assigned **by source** (no `-1/0/1`, no suspicious class — DEC-007 is moot here).

In [ ]:
from urllib.parse import urlparse

LABEL_PHISHING, LABEL_LEGIT = 1, 0

def load_phishing_urls() -> list[str]:
    """Read phishing URLs from any of the supported feeds in data/raw/."""
    # PhishTank online-valid CSV (has a 'url' column)
    p = RAW_DIR / "phishtank.csv"
    if p.exists():
        df = pd.read_csv(p)
        col = next((c for c in df.columns if c.lower() == "url"), df.columns[1])
        return df[col].dropna().astype(str).tolist()
    # OpenPhish feed (one URL per line)
    p = RAW_DIR / "phishing_urls.txt"
    if p.exists():
        return [l.strip() for l in p.read_text().splitlines() if l.strip().startswith("http")]
    # URLhaus recent CSV (commented header lines start with #; 'url' is a column)
    p = RAW_DIR / "urlhaus.csv"
    if p.exists():
        df = pd.read_csv(p, comment="#", header=None)
        # URLhaus column order: id,dateadded,url,url_status,...  → url is index 2
        return df[2].dropna().astype(str).tolist()
    raise FileNotFoundError(
        "No phishing feed found. Put one of: data/raw/phishtank.csv | phishing_urls.txt | urlhaus.csv"
    )

def load_legit_domains() -> list[str]:
    """Tranco top domains via the pip package, or a local CSV fallback."""
    try:
        from tranco import Tranco
        t = Tranco(cache=True, cache_dir=str(CACHE_DIR / "tranco"))
        lst = t.list(list_id=TRANCO_LIST_ID) if TRANCO_LIST_ID else t.list()
        print("Tranco list id:", lst.list_id)
        return [f"https://{d}/" for d in lst.top(N_PER_CLASS)]
    except Exception as e:
        print("tranco package unavailable (", e, ") — falling back to data/raw/tranco.csv")
        df = pd.read_csv(RAW_DIR / "tranco.csv", header=None, names=["rank", "domain"])
        return [f"https://{d}/" for d in df["domain"].head(N_PER_CLASS)]

def normalise(u: str) -> str | None:
    try:
        p = urlparse(u if "://" in u else "http://" + u)
        if p.scheme not in ("http", "https") or not p.netloc:
            return None
        host = p.netloc.lower()
        if STRIP_TO_HOSTNAME:
            return f"{p.scheme}://{host}/"
        return f"{p.scheme}://{host}{p.path}" + (f"?{p.query}" if p.query else "")
    except Exception:
        return None

phish = [n for n in (normalise(u) for u in load_phishing_urls()) if n]
legit = [n for n in (normalise(u) for u in load_legit_domains())  if n]

rng = np.random.default_rng(RANDOM_STATE)
def sample(xs, n):
    xs = list(dict.fromkeys(xs))  # dedupe, keep order
    return list(rng.choice(xs, size=min(n, len(xs)), replace=False)) if xs else []

phish, legit = sample(phish, N_PER_CLASS), sample(legit, N_PER_CLASS)
corpus = pd.DataFrame(
    {"url": phish + legit, "label": [LABEL_PHISHING]*len(phish) + [LABEL_LEGIT]*len(legit)}
).drop_duplicates(subset="url").reset_index(drop=True)
# Drop any URL that appears in BOTH classes (cross-class collision)
dupe = corpus.url[corpus.url.duplicated(keep=False)]
corpus = corpus[~corpus.url.isin(dupe)].reset_index(drop=True)
print(corpus.label.value_counts().rename({1: "phishing", 0: "legit"}))
corpus.head()

## 3. Extract features with **your** pipeline (cached)
Runs `src.features.pipeline.extract_features` over every URL in a thread pool. Dead/failed URLs are kept but flagged — never fatal (Iron Law #3). Results cache to parquet keyed by the config, so re-runs are instant.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

cache_key = hashlib.md5(
    f"{len(corpus)}-{FETCH_HTML}-{INCLUDE_DOMAIN}-{STRIP_TO_HOSTNAME}-{N_PER_CLASS}".encode()
).hexdigest()[:10]
cache_path = CACHE_DIR / f"features_{cache_key}.parquet"

def extract_row(rec):
    url, label = rec
    try:
        fv = extract_features(url, fetch_html=FETCH_HTML, include_domain=INCLUDE_DOMAIN)
        row = {k: v for k, v in fv.features.items() if not isinstance(v, str)}  # drop string categoricals
        row["url"] = url; row["label"] = label
        row["_ok"] = fv.model_version_compatible
        return row
    except Exception as e:
        return {"url": url, "label": label, "_ok": False, "_err": str(e)}

if cache_path.exists() and not FORCE_REEXTRACT:
    feats = pd.read_parquet(cache_path)
    print("loaded cached features:", cache_path.name)
else:
    t0 = time.perf_counter()
    rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = [ex.submit(extract_row, r) for r in corpus.itertuples(index=False, name=None)]
        for i, f in enumerate(as_completed(futs)):
            rows.append(f.result())
            if (i + 1) % 500 == 0:
                print(f"  {i+1}/{len(futs)} extracted")
    feats = pd.DataFrame(rows)
    feats.to_parquet(cache_path, index=False)
    print(f"extracted {len(feats)} in {time.perf_counter()-t0:.1f}s → {cache_path.name}")

print("reachable/usable rows:", int(feats['_ok'].sum()), "/", len(feats))
feats.head()

## 4. Build `X, y` from the model feature contract
Use the `feature_order` from `configs/feature_config.yaml`, then drop columns that are mostly empty (i.e. tiers you didn't run). This keeps training honest about which features actually exist.

In [ ]:
import yaml
cfg = yaml.safe_load((REPO_ROOT / "configs" / "feature_config.yaml").read_text())
contract = cfg["feature_order"]
MODEL_VERSION = cfg.get("model_version", "0.1.0")

present = [c for c in contract if c in feats.columns]
X_full = feats[present].apply(pd.to_numeric, errors="coerce")
# Drop features that are >50% missing (un-run tiers) — mirrors data_config max_null_rate
keep = [c for c in present if X_full[c].isna().mean() <= 0.50]
X = X_full[keep]
y = feats["label"].astype(int)
print(f"using {len(keep)}/{len(contract)} contract features:")
print(keep)
print("null rates:"); print(X.isna().mean().round(3).sort_values(ascending=False).head(10))

## 5. Split 70/15/15 (stratified, seed 42) — test set touched **once**, at the end

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE)
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5} n={len(yy):5d}  phishing%={yy.mean()*100:5.1f}")

## 6. Train DT / RF / LR / SVM — GridSearchCV, StratifiedKFold(5), refit on F1
Pipeline = `SimpleImputer → StandardScaler (LR/SVM only) → estimator`. Scaling is skipped for trees (they don't need it). `class_weight='balanced'` is the cheap imbalance fix (DEC-004).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def make_pipe(estimator, scale: bool):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("clf", estimator))
    return Pipeline(steps)

MODELS = {
    "decision_tree": (make_pipe(DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE), False),
                       {"clf__max_depth": [None, 10, 20], "clf__min_samples_leaf": [1, 5, 20]}),
    "random_forest": (make_pipe(RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1), False),
                       {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 20]}),
    "logistic_regression": (make_pipe(LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE), True),
                       {"clf__C": [0.1, 1.0, 10.0]}),
    "svm": (make_pipe(SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE), True),
                       {"clf__C": [1.0, 10.0], "clf__kernel": ["rbf"]}),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["f1", "precision", "recall", "roc_auc", "accuracy"]
fitted = {}
for name, (pipe, grid) in MODELS.items():
    t0 = time.perf_counter()
    gs = GridSearchCV(pipe, grid, scoring=scoring, refit="f1", cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train)   # CV on TRAIN ONLY
    fitted[name] = gs
    print(f"{name:20} cv_f1={gs.best_score_:.4f}  ({time.perf_counter()-t0:.1f}s)  best={gs.best_params_}")

## 7. Final evaluation on the held-out TEST set (once) + pick the winner
Best by **F1**, tie-break **ROC-AUC** (DEC-004). We emphasise the **false-negative rate** — a missed phishing site — since FNs are costlier.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
rows = []
for name, gs in fitted.items():
    proba = gs.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({
        "model": name,
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "accuracy": accuracy_score(y_test, pred),
        "fn_rate": fn / (fn + tp) if (fn + tp) else 0.0,  # missed phishing
    })
results = pd.DataFrame(rows).sort_values(["f1", "roc_auc"], ascending=False).reset_index(drop=True)
best_name = results.iloc[0]["model"]
print("BEST:", best_name)
results.round(4)

## 8. Save the production model (git-ignored — Iron Law #5)

In [ ]:
import joblib
best = fitted[best_name].best_estimator_
joblib.dump(best, MODELS_DIR / "model.joblib")
meta = {
    "model_name": best_name,
    "model_version": MODEL_VERSION,
    "features": keep,
    "tiers": {"html": FETCH_HTML, "domain": INCLUDE_DOMAIN},
    "test_metrics": results[results.model == best_name].iloc[0].to_dict(),
    "trained_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(MODELS_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2, default=str))
print("saved →", MODELS_DIR / "model.joblib")

## 9. SHAP — why the model decides what it decides (feeds `/explain` + thesis Ch.5)

In [ ]:
try:
    import shap
    Xt = best.named_steps["impute"].transform(X_test)
    if "scale" in best.named_steps:
        Xt = best.named_steps["scale"].transform(Xt)
    clf = best.named_steps["clf"]
    if best_name in ("decision_tree", "random_forest"):
        explainer = shap.TreeExplainer(clf)
        sv = explainer.shap_values(Xt)
        sv = sv[1] if isinstance(sv, list) else sv
    else:
        bg = shap.sample(Xt, 100, random_state=RANDOM_STATE)
        explainer = shap.KernelExplainer(clf.predict_proba, bg)
        sv = explainer.shap_values(Xt[:200])[1]
    shap.summary_plot(sv, features=Xt[:sv.shape[0]], feature_names=keep, show=True)
except Exception as e:
    print("SHAP skipped:", e)

## 10. Next steps
1. **Sanity-check for the path-bias artifact** (top of notebook): if `url_length`/`path_length` dominate the SHAP plot, that's likely the PhishTank-vs-Tranco collection artifact, not real signal. Re-run with `STRIP_TO_HOSTNAME = True` and compare — a big metric drop means you were leaking. Document it.
2. **Enrich:** flip `FETCH_HTML`/`INCLUDE_DOMAIN = True` for the HTML + WHOIS/DNS tiers (slower, more PhishTank attrition). Compare lexical-only vs enriched in `EXPERIMENTS.md`.
3. **Scale up** `N_PER_CLASS` once the pipeline is trusted.
4. **Productionise** this into `scripts/train.py` / `src/models/` per Phase 5 of the guide; record results (append-only) in `EXPERIMENTS.md` and the best-model choice as a new `DEC-0xx`.
5. The saved `models/production/model.joblib` is what Phase 6's `/detect` loads at startup.